# Um token opcional para os rates de download

In [1]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

load_dotenv()
token = os.getenv('HF_TOKEN_READ')

if token is None:
    print("HF_TOKEN_READ não encontrado no .env — a continuar sem token")
else:
    login(token=token)
    print("Token carregado e sessão HF iniciada")

Token carregado e sessão HF iniciada


# Imports

In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset

# Load e Processamento dos dados

### Escolher apenas textos entre 80 e 150 palavras

In [3]:
def count_rows_by_word_range(df, text_col, min_words=80, max_words=150):
    word_counts = df[text_col].fillna('').apply(lambda x: len(str(x).split())) 
    mask = (word_counts > min_words) & (word_counts < max_words)
    total = len(df)
    filtered = mask.sum()
    print(f"Total de linhas: {total}")
    print(f"Linhas com >{min_words} e <{max_words} palavras: {filtered}")
    print(f"Percentagem: {filtered/total*100:.1f}%")
    return df[mask]

### Coluna conversations, formato comum de guardar conversas de LLMs

In [4]:
import ast

def extract_gpt_texts(df, col='conversations'):
    texts = []
    for convo in df[col]:
        # se vier como string, converte para lista
        if isinstance(convo, str):
            convo = ast.literal_eval(convo)
        for turn in convo:
            if turn.get('from') == 'gpt':
                texts.append(turn['value'])
    return texts

## Anthropic

### Dataset que contém vários datasets

In [5]:
df1_anthropic = load_dataset('agentlans/claude', name="all", split='train').to_pandas()

In [6]:
df1_anthropic_texts = extract_gpt_texts(df1_anthropic)
df1_anthropic_text_c = pd.DataFrame({'text': df1_anthropic_texts})

print(f"Total de respostas GPT: {len(df1_anthropic_text_c)}")

Total de respostas GPT: 977918


In [7]:
df1_anthropic_f = count_rows_by_word_range(df1_anthropic_text_c, text_col='text')

Total de linhas: 977918
Linhas com >80 e <150 palavras: 145757
Percentagem: 14.9%


In [8]:
df1_anthropic_f["label"] = "Anthropic"

In [9]:
df1_anthropic_f.head()

,text,label
1,Tropical fruit: Mangoes. A sweet fleshy fruit ...,Anthropic
9,Here is an Excel table comparing the verb form...,Anthropic
13,A potential subject line for this email could ...,Anthropic
21,"Sure, I can help you create a static HTML webp...",Anthropic
24,B. No\n\nLet me explain why this is not a good...,Anthropic


In [10]:
df1_anthropic_f.info()

<class 'pandas.DataFrame'>
Index: 145757 entries, 1 to 977912
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    145757 non-null  str  
 1   label   145757 non-null  str  
dtypes: str(2)
memory usage: 107.8 MB


## Google

### Dataset grande

In [11]:
df1_google = load_dataset('mlfoundations-dev/oh-dcft-v3.1-gemini-1.5-pro', split='train').to_pandas()

In [12]:
df1_google_conversations_texts = extract_gpt_texts(df1_google)
df1_google_text = pd.DataFrame({'text': df1_google_conversations_texts})

In [13]:
df1_google_f = count_rows_by_word_range(df1_google_text, text_col='text')

Total de linhas: 1001459
Linhas com >80 e <150 palavras: 101983
Percentagem: 10.2%


In [14]:
df1_google_f["label"] = "Google"

In [15]:
df1_google_f.head()

,text,label
27,There's no single fact that *only* Elon Musk's...,Google
28,A Brew of Black – to wake the Soul – \nA bitte...,Google
30,A Curve – a fragile Infinity – \nA Number's hu...,Google
31,* **Americas:**\n * Christopher Columbus (C...,Google
33,Could a vacation *be* any more perfect? Pictu...,Google


In [16]:
df1_google_f.info()

<class 'pandas.DataFrame'>
Index: 101983 entries, 27 to 1001452
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    101983 non-null  str  
 1   label   101983 non-null  str  
dtypes: str(2)
memory usage: 67.5 MB


## Meta

### Dataset de 1M de linhas Llama 3

In [17]:
df1_meta = load_dataset('Magpie-Align/Magpie-Llama-3.3-Pro-1M-v0.1', split='train').to_pandas()

In [18]:
df1_meta = df1_meta[['response']]
df1_meta.rename(columns={'response': 'text'}, inplace=True)

In [19]:
df1_meta_f = count_rows_by_word_range(df1_meta, text_col='text')

Total de linhas: 1000000
Linhas com >80 e <150 palavras: 87501
Percentagem: 8.8%


In [20]:
df1_meta_f["label"] = "Meta"

In [21]:
df1_meta_f.head()

,text,label
0,Here's how you can achieve this using list com...,Meta
3,I'm happy to do either. I can engage in a wide...,Meta
12,## Step 1: Write down the given equation\nThe ...,Meta
20,I'd be happy to help you narrow down your inte...,Meta
53,"Yes, in this context, ""in place of"" is an idio...",Meta


In [22]:
df1_meta_f.info()

<class 'pandas.DataFrame'>
Index: 87501 entries, 0 to 999988
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    87501 non-null  str  
 1   label   87501 non-null  str  
dtypes: str(2)
memory usage: 57.8 MB


## OpenAI

### Mistura de vários datasets

In [23]:
df1_openai = load_dataset('agentlans/chatgpt', name='all', split='train').select_columns(['output']).to_pandas().rename(columns={'output': 'text'}).reset_index(drop=True)

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [24]:
df1_openai_f = count_rows_by_word_range(df1_openai, text_col='text')

Total de linhas: 3281633
Linhas com >80 e <150 palavras: 504343
Percentagem: 15.4%


### Retirar linguagens que n sejam inglês, mesmo que a quantidade seja muito pequena

In [25]:
import re

def is_mostly_english(text):
    non_latin = len(re.findall(r'[^\x00-\x7F]', str(text)))
    total = len(str(text))
    return (non_latin / total) < 0.1  # menos de 10% de caracteres não-ASCII

mask = df1_openai_f['text'].apply(is_mostly_english)
df1_openai_f = df1_openai_f[mask].reset_index(drop=True)
print(f"Após filtro: {len(df1_openai_f)}")

Após filtro: 498253


In [26]:
df1_openai_f["label"] = "Openai"

In [27]:
df1_openai_f.head()

,text,label
0,"Alright, let me explain it in a way you might ...",Openai
1,If you are referring to one of the notably thi...,Openai
2,"In the United States, student privacy is guard...",Openai
3,The poem mentions four specific nutrients that...,Openai
4,In the context of the assembly elections in Hi...,Openai


In [28]:
df1_openai_f.info()

<class 'pandas.DataFrame'>
RangeIndex: 498253 entries, 0 to 498252
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    498253 non-null  str  
 1   label   498253 non-null  str  
dtypes: str(2)
memory usage: 340.8 MB


## Human

### Dataset de comparação de vários modelos de IA com humanos

In [29]:
df1_human = load_dataset('liamdugan/raid', split='train').select_columns(['model', 'generation']).to_pandas().rename(columns={'generation': 'text'}).reset_index(drop=True)

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

In [30]:
df1_human = df1_human[df1_human['model'] == 'human'][['text']].reset_index(drop=True)

df1_human_f = count_rows_by_word_range(df1_human, text_col='text')

Total de linhas: 160452
Linhas com >80 e <150 palavras: 15418
Percentagem: 9.6%


In [31]:
df1_human_f["label"] = "Human"

In [32]:
df1_human_f.head()

,text,label
1,High-quality training data play a key role in ...,Human
18,"We introduce $\textit{InExtremIS}$, a weakly s...",Human
20,Fully convolutional U-shaped neural networks h...,Human
48,Despite the remarkable performance of deep lea...,Human
84,We extend first-order model agnostic meta-lear...,Human


In [33]:
df1_human_f.info()

<class 'pandas.DataFrame'>
Index: 15418 entries, 1 to 158219
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    15418 non-null  str  
 1   label   15418 non-null  str  
dtypes: str(2)
memory usage: 15.2 MB


### Dataset comparação IA vs Humano

In [34]:

df2_human = load_dataset('dmitva/human_ai_generated_text', split='train').select_columns(['human_text']).to_pandas().rename(columns={'human_text': 'text'}).reset_index(drop=True)


In [35]:
df2_human_f = count_rows_by_word_range(df2_human, text_col='text')

Total de linhas: 1000000
Linhas com >80 e <150 palavras: 13160
Percentagem: 1.3%


In [36]:
df2_human_f['label'] = 'Human'

In [37]:
df2_human_f.head()

,text,label
335,Students need their phones for any emergencies...,Human
336,"TOMAS JEFFERSON\n\nThomas Jefferson wrote, imp...",Human
485,Ether just because of that resounds that they ...,Human
500,One way that the principal should decide is by...,Human
702,"Dear Principal,\n\nThe reason why I'm writing ...",Human


In [38]:
df2_human_f.info()

<class 'pandas.DataFrame'>
Index: 13160 entries, 335 to 999952
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    13160 non-null  str  
 1   label   13160 non-null  str  
dtypes: str(2)
memory usage: 9.4 MB


### Mais um dataset de humano vs IA

In [39]:
df3_human = load_dataset('ahmadreza13/human-vs-Ai-generated-dataset', split='train').to_pandas().rename(columns={'data': 'text'}).reset_index(drop=True)

In [40]:
df3_human = df3_human[df3_human['model'] == "wikipedia"][['text']].reset_index(drop=True)
df3_human_f = count_rows_by_word_range(df3_human, text_col='text')

Total de linhas: 2092358
Linhas com >80 e <150 palavras: 654364
Percentagem: 31.3%


In [41]:
df3_human_f["label"] = "Human"

In [42]:
df3_human_f.head()

,text,label
1,Military academy\n\nA military academy or serv...,Human
7,"Motorola\n\nMotorola, Inc. () was an American ...",Human
20,Mountaineering\n\nMountaineering is the set of...,Human
28,Mozambique Channel\n\nThe Mozambique Channel (...,Human
29,Medical psychology\n\nMedical psychology is th...,Human


In [43]:
df3_human_f.info()

<class 'pandas.DataFrame'>
Index: 654364 entries, 1 to 2092355
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    654364 non-null  str  
 1   label   654364 non-null  str  
dtypes: str(2)
memory usage: 447.3 MB


# Sample para o dataset final

In [44]:
seed = 8088135
df_openai_final = df1_openai_f.sample(n=50000, random_state=seed)
df_google_final = df1_google_f.sample(n=50000, random_state=seed)
df_meta_final = df1_meta_f.sample(n=50000, random_state=seed)
df_anthropic_final = df1_anthropic_f.sample(n=50000, random_state=seed)

df_human_all = pd.concat([df1_human_f, df2_human_f], ignore_index=True)
n_others = len(df_human_all)
n_wiki = 50000 - n_others
df_wikipedia_sample = df3_human_f.sample(n=n_wiki, random_state=seed)
df_human_final = pd.concat([df_human_all, df_wikipedia_sample], ignore_index=True)

print(f"human outros: {n_others} | wikipedia: {n_wiki}")
print(f"Total humano: {len(df_human_final)}")

df_final = pd.concat([df_openai_final, df_google_final, df_meta_final, 
                      df_anthropic_final, df_human_final], ignore_index=True)
print(f"Total final: {len(df_final)}")
print(df_final['label'].value_counts())

human outros: 28578 | wikipedia: 21422
Total humano: 50000
Total final: 250000
label
Openai       50000
Google       50000
Meta         50000
Anthropic    50000
Human        50000
Name: count, dtype: int64


## Upload do dataset no hugging face

In [45]:
from dotenv import load_dotenv
import os

load_dotenv()
token = os.getenv('HF_TOKEN_WRITE')

from huggingface_hub import login

login(token=token)

In [46]:
from datasets import Dataset

dataset = Dataset.from_pandas(df_final)
dataset.push_to_hub('BrunoFilipeRDS/50k-balanced-5-classes', token=token)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/BrunoFilipeRDS/50k-balanced-5-classes/commit/91f00e9f84c658c134cfdd3c70ab71cbc6e9635e', commit_message='Upload dataset', commit_description='', oid='91f00e9f84c658c134cfdd3c70ab71cbc6e9635e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/BrunoFilipeRDS/50k-balanced-5-classes', endpoint='https://huggingface.co', repo_type='dataset', repo_id='BrunoFilipeRDS/50k-balanced-5-classes'), pr_revision=None, pr_num=None)